In [1]:
import os
import json
import requests
from typing import TypedDict, Annotated, List, Literal
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel, Field
from IPython.display import Image, display

load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_REPO  = "rahul8879/e-comm-agentic-demo"
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
BASE_URL = f"https://api.github.com/repos/{GITHUB_REPO}"

llm = ChatOpenAI(model="gpt-4o", temperature=0)

r = requests.get(BASE_URL, headers=HEADERS)
print(f"Repo: {r.json().get('full_name')}")
print(f"Status: {r.status_code}")

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo: rahul8879/e-comm-agentic-demo
Status: 200


In [2]:
def get_pr_details(pr_number: int) -> dict:
    r = requests.get(f"{BASE_URL}/pulls/{pr_number}", headers=HEADERS)
    data = r.json()
    return {
        "title":  data.get("title"),
        "author": data.get("user", {}).get("login"),
        "body":   data.get("body") or "No description provided.",
        "branch": data.get("head", {}).get("ref"),
    }

def get_pr_files(pr_number: int) -> list:
    r = requests.get(f"{BASE_URL}/pulls/{pr_number}/files", headers=HEADERS)
    files = r.json()
    return [
        {"filename": f["filename"], "patch": f.get("patch", "")}
        for f in files
    ]

def post_pr_comment(pr_number: int, comment_body: str) -> str:
    url = f"{BASE_URL}/issues/{pr_number}/comments"
    r = requests.post(url, headers=HEADERS, json={"body": comment_body})
    return "posted" if r.status_code == 201 else f"failed: {r.status_code}"

In [ ]:
get_pr_files(11)[0]['patch'][:500]

IndexError: list index out of range

In [ ]:
# get_pr_files(11)

# whenever you start building any agentic systems ---> ccc
# communications --> blackboard/or shared state
class ReviewState(TypedDict):
    pr_number: int
    pr_details: dict
    pr_files: list
    plan: List[str]
    security_findings: str
    style_findings: str
    test_findings: str
    docs_findings: str          # NEW
    performance_findings: str   # NEW
    final_comment: str
    approved: bool


In [6]:
# step 2 : define my manager ? superviser agent / lead agent
# definition ---> llm + prompt +memory + tools-->output format ?? 

class ReviewPlan(BaseModel):
    needs_security: bool = Field(description="True if this PR touches auth, payments, SQL, secrets, or user input handling")
    needs_style: bool = Field(description="True if this PR changes naming, formatting, or structure worth a style pass")
    needs_tests: bool = Field(description="True if this PR adds or changes logic that should have test coverage")
    needs_docs: bool = Field(description="True if this PR adds public functions/APIs that need documentation")
    needs_performance: bool = Field(description="True if this PR touches loops, DB queries, or anything with performance implications")
    reason: str = Field(description="One short sentence explaining the routing decision")

planner_llm = llm.with_structured_output(ReviewPlan)


In [12]:
def manager_node(state:ReviewState):
    pr_details =get_pr_details(state["pr_number"])
    pr_files = get_pr_files(state["pr_number"])

    summary = "\n\n".join([f"File: {f['filename']}\nPatch:\n{f['patch'][:500]}" for f in pr_files])

    plan: ReviewPlan = planner_llm.invoke([
        SystemMessage(content=(
            "You are the lead reviewer on CodeSentinel. Look at this PR's diff "
            "and decide which specialist reviewers actually need to look at it. "
            "Don't call a specialist unless their concern is genuinely relevant."
        )),
        HumanMessage(content=f"PR title: {pr_details['title']}\n\nDiff:\n{summary}")

    ])

    return plan


In [13]:
test_state = {
    "pr_number": 1, "pr_details": {}, "pr_files": [], "plan": [],
    "security_findings": "", "style_findings": "", "test_findings": "",
    "docs_findings": "", "performance_findings": "",
    "final_comment": "", "approved": False,
}

manager_node(test_state)

ReviewPlan(needs_security=True, needs_style=False, needs_tests=True, needs_docs=False, needs_performance=False, reason='The PR introduces security vulnerabilities and lacks test coverage for critical functions.')